# D2 · Spectral indices: vegetation, water and bare ground in the Melamchi valley

> **Time:** about 2.5 hours  
> **You need:** Parts A, B and C, especially functions (A5), the integer overflow trap and masks (C5) and `reproject_match` (C4), and Lesson D1: bands, reflectance, composites and spectral profiles.  
> **By the end you can:**
> 1. do band maths safely: convert to floats before subtracting, turn nodata into `NaN`, and guard against division by zero,
> 2. write one function for any **normalised difference**, and use it for **NDVI**, **NDWI** and **MNDWI**, with maps that use diverging colours centred on 0,
> 3. turn an index into a mask and an area in km², compare the two dates by elevation using the DEM, and save an index as a GeoTIFF.

In D1 you saw that each surface has its own spectral profile: forest is dark in red and bright in NIR, sediment is bright everywhere, water is dark in NIR. A **spectral index** squeezes such a difference into **one number per cell**. One map then shows where vegetation, water or bare ground is.

This lesson builds the most widely used indices for the two Sentinel-2 images of the Melamchi valley, and compares their statistics before and after the June 2021 flood. Mapping the change itself, cell by cell, is the subject of D3.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.* This credit line must appear wherever the images, or maps made from them, are shown.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** reprojecting the DEM (step 6) can differ very slightly between library versions, so your decimals there may differ a little from the text.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 31 of 32 · [Course home](../README.md)  
← Previous: [D1 · Satellite images and spectral bands: the Melamchi valley from space](D01_Satellite_images_and_spectral_bands.ipynb) · Next: [D3 · Change detection: the 2021 Melamchi flood](D03_Change_detection_Melamchi_flood.ipynb) →

## 1. Band maths needs floats: the `uint16` trap

**Band maths** means arithmetic on bands, cell by cell: subtract one band from another, divide, add. It works like the grid arithmetic of C5.

The files store bands as `uint16` (D1): whole numbers from 0 to 65,535, with **no negative numbers**. Over vegetation, red minus NIR is negative. In `uint16` a negative result cannot exist, so it **wraps around** from 65,535 downwards, as `int16` overflowed in C5 and `uint8` in A8. NumPy gives no error.

The fix is the one you know: convert to a float **before** you calculate, with `.astype("float32")`, then scale to reflectance.

**Predict:** what will red − NIR give for a forest cell with a red value of 300 and a NIR value of 2,900?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import rioxarray as rxr
from rasterio.enums import Resampling

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
before_path = DATA / "s2_melamchi_before.tif"
after_path = DATA / "s2_melamchi_after.tif"

with rasterio.open(before_path) as src:
    red_dn = src.read(3)       # B04, uint16
    nir_dn = src.read(7)       # B08, uint16

wrong = red_dn - nir_dn        # uint16 - uint16 = uint16
print(wrong.dtype, "| lowest:", wrong.min(), "| highest:", wrong.max())
print("one forest cell: red", red_dn[296, 537], "NIR", nir_dn[296, 537], "-> red - NIR =", wrong[296, 537])
print("cells where red < NIR:", (red_dn < nir_dn).sum())

right = red_dn.astype("float32") - nir_dn.astype("float32")
print(right.dtype, "| same cell:", right[296, 537], "| lowest:", right.min())

In [ ]:
def open_s2(path):
    """Open a course Sentinel-2 file as reflectance (float32, NaN for nodata), with named bands, as in D1."""
    image = rxr.open_rasterio(path, masked=True)
    image = image.assign_coords(band=list(image.attrs["long_name"])) * 0.0001
    image.attrs = {}      # the file's old notes (scale factor, band names) no longer describe these values
    return image

before = open_s2(before_path)
after = open_s2(after_path)
print(before.dtype, before.shape, before.band.values)

left, bottom, right_m, top = before.rio.bounds()
extent_km = [left / 1000, right_m / 1000, bottom / 1000, top / 1000]
CELL_KM2 = 20 * 20 / 1000000          # one 20 m cell is 400 m² = 0.0004 km²

The forest cell from D1's spectral profiles has a red value of 321 and a NIR value of 4,130. In `uint16`, 321 − 4,130 gives **61,727**: the result wrapped round from 65,536. The wrapped grid runs from 0 to 65,535, with no negative value anywhere. Red is below NIR in 705,813 cells, so almost every cell of the image is wrong.

In `float32` the same cell gives −3,809, as it should, and the lowest difference is −13,743.

The second cell puts D1's opening steps into a function (A5), so each image takes one line. The function also clears `.attrs`. xarray carries a file's attributes along with your results, including `scale_factor` 0.0001. If you later saved an index with that note, a program reading the file would scale it a second time.

`CELL_KM2` turns cell counts into km², as in C5: 20 m × 20 m = 400 m².

### Practice

**Q1 · one line:** Repeat the trap on two tiny arrays: subtract `np.array([2900], dtype="uint16")` from `np.array([300], dtype="uint16")`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.array([300], dtype="uint16") - np.array([2900], dtype="uint16")
```

62,936 instead of −2,600: 65,536 − 2,600. Every negative result is shifted up by 65,536.
</details>

**Q2 · two lines:** Is **adding** the two bands safe in `uint16`? Compute `red_dn + nir_dn` in `uint16` and in `float32`, and print the highest value of each.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((red_dn + nir_dn).max())
print((red_dn.astype("float32") + nir_dn.astype("float32")).max())
```

Both give 18,838, so this sum fits below 65,535. Addition is safe here only by luck of the data. Convert first anyway: one rule for all band maths is easier to keep than a rule per operation.
</details>

**Q3 · three lines:** What share of the wrapped grid `wrong` lies above 60,000, a value no real difference of reflectances could take? Print the count and the percentage of all cells, rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
big = (wrong > 60000).sum()
print(big)
print(round(big / wrong.size * 100, 1))
```

702,989 cells, about 99.5% of the image, hold impossible values above 60,000. A quick look at the range of a result catches the trap at once.
</details>

## 2. One function for every normalised difference

Most indices have the same shape, a **normalised difference** of two bands:

**(A − B) / (A + B)**

Dividing by the sum makes the result independent of how bright a cell is overall. Shadowed and sunlit slopes of the same forest get similar values. The result always lies between **−1** and **+1** for reflectances of 0 or more.

One function (A5) serves every index. It needs one guard: if A + B is 0, the division is 0 / 0. The function replaces zero sums with `NaN` first, so the answer is `NaN`, the honest "no answer".

There is a second trap: **nodata**. A cell with red = 0 (nodata) and NIR = 2,000 gives (2,000 − 0) / (2,000 + 0) = 1, a perfect index value for a cell with no data. Nodata must become `NaN` **before** the index, as `masked=True` does in `open_s2`.

In [ ]:
def normalized_difference(a, b):
    """(a - b) / (a + b) for each cell, with NaN where a + b is 0."""
    total = a + b
    total = np.where(total == 0, np.nan, total)     # guard: no division by zero
    return (a - b) / total

# Test on made-up reflectances: forest-like, equal bands, both zero, more b than a
print(normalized_difference(np.array([0.40, 0.30, 0.00, 0.10]), np.array([0.04, 0.30, 0.00, 0.30])))

In [ ]:
# Nodata on purpose: the raw after image, without masked=True
with rasterio.open(after_path) as src:
    red_raw = src.read(3).astype("float32")
    nir_raw = src.read(7).astype("float32")

raw_index = normalized_difference(nir_raw, red_raw)
print("cells equal to exactly 1:", (raw_index == 1).sum(), "| exactly -1:", (raw_index == -1).sum())

masked_index = normalized_difference(after.sel(band="B08"), after.sel(band="B04"))
print("with masked=True -> NaN cells:", int(masked_index.isnull().sum()), "| cells equal to 1:", int((masked_index == 1).sum()))

The test gives about 0.818 for the forest-like pair, 0 for equal bands, `nan` for 0 and 0 (no warning, no crash), and −0.5 when b is larger. Testing a function on values whose answers you know is the habit from C6.

On the raw after image, 125 cells come out as exactly **1** and 3 as exactly **−1**. They are the shadow cells of D1 with red = 0 or NIR = 0. Their perfect-looking values are artefacts of nodata. With `masked=True`, those 128 cells are `NaN`, and no cell equals 1.

The function works on NumPy arrays and on DataArrays alike. With DataArrays, the result keeps the coordinates and the CRS, so it is still a map.

### Practice

**Q1 · one line:** Use `normalized_difference` on two numbers in arrays: a = 0.25, b = 0.05.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
normalized_difference(np.array([0.25]), np.array([0.05]))
```

(0.25 − 0.05) / (0.25 + 0.05) = 0.20 / 0.30 ≈ 0.667.
</details>

**Q2 · two lines:** The scale factor cancels: compare the index from stored values (`nir_dn`, `red_dn` of step 1, as `float32`) with the index from reflectances (`before`). Print the largest absolute difference between the two.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
from_dn = normalized_difference(nir_dn.astype("float32"), red_dn.astype("float32"))
print(np.abs(from_dn - normalized_difference(before.sel(band="B08"), before.sel(band="B04")).values).max())
```

About 1.19e-07, that is 0.000000119, which comes only from rounding in `float32` arithmetic. 0.0001 appears above and below the line and cancels. This works only because the offset is 0: with the newer offset (D1), stored values must be converted to reflectance first.
</details>

**Q3 · three lines:** Show what goes wrong **without** the guard: divide `np.array([0.0])` minus itself by `np.array([0.0])` plus itself, inside `with np.errstate(invalid="raise"):` so that NumPy raises an error instead of a warning, and catch it with `try`/`except FloatingPointError` (A6).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
try:
    with np.errstate(invalid="raise"):
        (np.array([0.0]) - np.array([0.0])) / (np.array([0.0]) + np.array([0.0]))
except FloatingPointError as err:
    print("FloatingPointError:", err)
```

NumPy reports an **invalid value encountered in divide**: 0 / 0. Normally it only prints a warning and returns `nan`, which you can miss in a long notebook. The guard in `normalized_difference` makes the `NaN` deliberate.
</details>

## 3. NDVI: how green is each cell?

The **Normalised Difference Vegetation Index** (NDVI, Rouse and others, 1974) compares NIR with red:

**NDVI = (NIR − red) / (NIR + red)**, for Sentinel-2 **(B08 − B04) / (B08 + B04)**.

Green leaves absorb red light and reflect NIR strongly (the red edge of D1), so they give high values. A rough guide, not a fixed rule:

| NDVI | Usually |
| --- | --- |
| below 0 | water, snow, cloud |
| about 0 to 0.2 | bare rock, gravel, sediment, buildings |
| about 0.2 to 0.5 | sparse or dry vegetation, fields, mixed cells |
| above about 0.6 | dense green vegetation, such as forest |

Indices run from −1 to +1 with **0** as a natural middle, so maps use a **diverging** colour map centred on 0. With `cmap="RdYlGn"`, `vmin=-1` and `vmax=1`, red means negative, yellow 0, and green positive. Always give both dates the **same** `vmin` and `vmax`.

In [ ]:
ndvi_before = normalized_difference(before.sel(band="B08"), before.sel(band="B04"))
ndvi_after = normalized_difference(after.sel(band="B08"), after.sel(band="B04"))

for name, ndvi in [("before", ndvi_before), ("after", ndvi_after)]:
    print(f"{name:6}  mean {float(ndvi.mean()):.3f}  median {float(ndvi.median()):.3f}  "
          f"min {float(ndvi.min()):.3f}  max {float(ndvi.max()):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 6.8))
fig.subplots_adjust(wspace=0.3)
for ax, ndvi, title in zip(axes, [ndvi_before, ndvi_after], ["NDVI, 29 October 2020", "NDVI, 11 November 2021"]):
    im = ax.imshow(ndvi, extent=extent_km, cmap="RdYlGn", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.colorbar(im, ax=axes, label="NDVI", shrink=0.7)
fig.text(0.01, 0.01, "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.", fontsize=9)
plt.show()

In [ ]:
# NDVI at D1's four cells (forest, built-up area, river from 2020; sediment from 2021)
places = {"forest": (ndvi_before, 361010, 3089990),
          "built-up area": (ndvi_before, 359610, 3079190),
          "river, water and gravel": (ndvi_before, 359710, 3076730),
          "flood sediment": (ndvi_after, 359610, 3077990)}
for name, (ndvi, x_m, y_m) in places.items():
    print(f"{name:24} {float(ndvi.sel(x=x_m, y=y_m, method='nearest')):.3f}")

The median NDVI is about 0.82 before and 0.81 after: most of the valley is densely vegetated on both dates. The means are lower, about 0.765 and 0.738, pulled down by the river bed, fields and settlements.

On the maps the hills are deep green. The valley floor stands out in pale green and yellow, and in 2021 a wide yellow strip follows the river: sediment with almost no vegetation. Few cells are red, because there is little open water.

The extremes come from deep shadow. Where red is close to 0, NDVI approaches 1 (the maximum before is about 0.999). The lowest values, down to about −0.69 after, are mixed cells of water and wet gravel.

The four cells of D1 fit the guide: forest about 0.856, built-up area about 0.07, river cell about −0.23, and flood sediment about 0.03.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** Read the NDVI of the sediment cell (359610, 3077990) in the **before** image with `.sel(..., method="nearest")`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
float(ndvi_before.sel(x=359610, y=3077990, method="nearest"))
```

About 0.58: in 2020 this cell had vegetation (D1, step 7, Q3). By November 2021 its NDVI had fallen to about 0.03.
</details>

**Q2 · two lines:** What share of the before image counts as dense vegetation, NDVI above 0.6? Divide the count by the number of cells with a value (`ndvi_before.notnull().sum()`) and print the percentage rounded to 1 decimal. Then do the same for after.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(float((ndvi_before > 0.6).sum() / ndvi_before.notnull().sum() * 100), 1))
print(round(float((ndvi_after > 0.6).sum() / ndvi_after.notnull().sum() * 100), 1))
```

About 86.5% before and 83.0% after: 3.5 percentage points less. Part of that is the flood; part is the later date, when fields were drier and shadows longer.
</details>

**Q3 · three lines:** Draw NDVI after with a colour map that is **not** centred on 0: `vmin=0.2`, `vmax=1`. Compare it with the example map. What changes?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(ndvi_after, extent=extent_km, cmap="RdYlGn", vmin=0.2, vmax=1)
fig.colorbar(im, ax=ax, label="NDVI", shrink=0.7)
ax.set_title("NDVI, 11 Nov 2021, NOT centred on 0")
plt.show()
```

Now 0.6, healthy vegetation, sits at yellow in the middle of the colour map, and the valley floor and many fields turn deep red or orange, as if it were water or something alarming. Diverging colours suggest "below or above a meaningful middle"; if the middle is not 0, the map tells a different story than the numbers.
</details>

## 4. Histograms, thresholds and areas

A map shows **where**; a **histogram** shows **how much** of each value (C2). For an index, the histogram shows its usual values and the tail of unusual ones.

To turn an index into an answer you choose a **threshold**: a value that splits cells into two groups, for example "NDVI below 0.3: little or no vegetation". The comparison gives a True/False **mask**, as in C5. Its sum is a number of cells, and cells × 0.0004 km² is an area.

A threshold is **your choice**, not a fact of nature. Always say which one you used, and look at the histogram first. D3 shows how much an area moves when the threshold moves.

`ax.hist` needs the values as one flat list of numbers without `NaN`: `ndvi.values[np.isfinite(ndvi.values)]` keeps only real numbers (C5).

In [ ]:
values_before = ndvi_before.values[np.isfinite(ndvi_before.values)]
values_after = ndvi_after.values[np.isfinite(ndvi_after.values)]

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.arange(-0.7, 1.01, 0.02)
ax.hist(values_before, bins=bins, color="seagreen", alpha=0.6, label="29 October 2020")
ax.hist(values_after, bins=bins, color="darkorange", alpha=0.6, label="11 November 2021")
ax.axvline(0.3, color="black", linestyle="--", label="threshold 0.3")
ax.set_yscale("log")                        # small counts would be invisible on a normal scale
ax.set_xlabel("NDVI")
ax.set_ylabel("Number of cells (log scale)")
ax.set_title("NDVI of the Melamchi valley on two dates")
ax.legend()
plt.show()

In [ ]:
THRESHOLD = 0.3
for name, ndvi in [("before", ndvi_before), ("after", ndvi_after)]:
    low = ndvi < THRESHOLD                      # True/False mask, as in C5
    cells = int(low.sum())
    print(f"{name:6} NDVI < {THRESHOLD}: {cells:,} cells = {cells * CELL_KM2:.2f} km²")

fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(ndvi_after < THRESHOLD, extent=extent_km, cmap="Greys")
ax.set_title("NDVI below 0.3, 11 November 2021 (black)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

Both histograms have one large peak between about 0.8 and 0.9: the forest and green fields. On the log scale you can see the long tail towards 0 and below. The 2021 histogram has **more cells in the tail** below 0.3, with a second, small peak slightly above 0: the sediment.

With NDVI < 0.3 as the rule, 8,606 cells (about 3.44 km²) had little or no vegetation in October 2020, and 19,627 cells (about 7.85 km²) in November 2021. The mask map shows where: a continuous black strip along the Melamchi river, thinner lines along the other two rivers, and small specks of fields and settlements on the slopes.

`NaN < 0.3` is `False`, so nodata cells never enter the count. The log scale (`set_yscale("log")`) gives each power of ten the same height, so a bar of 10 cells and a bar of 100,000 cells are both visible.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How many cells of `ndvi_after` are negative? Use a mask and `.sum()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int((ndvi_after < 0).sum())
```

2,176 cells (0.87 km²), against 398 before. Negative NDVI means more red than NIR: water and wet gravel, which the flood spread across the valley floor.
</details>

**Q2 · two lines:** Try a stricter threshold: count the cells below **0.2** on both dates and print the areas in km², rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for name, ndvi in [("before", ndvi_before), ("after", ndvi_after)]:
    print(name, round(int((ndvi < 0.2).sum()) * CELL_KM2, 2))
```

About 1.67 km² before and 5.16 km² after, instead of 3.44 and 7.85 with 0.3. The areas depend strongly on the threshold, but the after image has much more bare ground with either rule.
</details>

**Q3 · three lines:** Use `np.percentile` on `values_before` and `values_after` to print the 2nd, 50th and 98th percentiles of NDVI on each date, rounded to 3 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(np.round(np.percentile(values_before, [2, 50, 98]), 3))
print(np.round(np.percentile(values_after, [2, 50, 98]), 3))
```

The medians (about 0.82 and 0.81) and the 98th percentiles (about 0.91 and 0.90) barely move. The 2nd percentile drops from about 0.35 to about 0.23. The flood changed a small part of the image a lot, so it shows in the low tail, not in the middle.
</details>

## 5. Water and wet sediment: NDWI and MNDWI

Water reflects a little green light, almost no NIR and even less SWIR (D1). Two indices use that:

- **NDWI** (McFeeters, 1996) = (green − NIR) / (green + NIR) = **(B03 − B08) / (B03 + B08)**.
- **MNDWI**, the modified NDWI (Xu, 2006) = (green − SWIR 1) / (green + SWIR 1) = **(B03 − B11) / (B03 + B11)**. SWIR separates water from built-up areas and shadow better than NIR.

Values **above 0** usually mean water or very wet ground. Vegetation gives strongly negative values. Careful with names: another index, also called NDWI (Gao, 1996), uses NIR and SWIR to measure water inside leaves. Always give the formula with the name.

The Melamchi river is narrow at 20 m (D1, review 2), so expect few cells above 0. The example zooms in on the place where the rivers meet with `clip_box` (C3).

In [ ]:
ndwi = {}
mndwi = {}
for name, image in [("before", before), ("after", after)]:
    ndwi[name] = normalized_difference(image.sel(band="B03"), image.sel(band="B08"))
    mndwi[name] = normalized_difference(image.sel(band="B03"), image.sel(band="B11"))
    print(f"{name:6} NDWI > 0: {int((ndwi[name] > 0).sum()):5,} cells = {int((ndwi[name] > 0).sum()) * CELL_KM2:.2f} km²"
          f" | MNDWI > 0: {int((mndwi[name] > 0).sum()):5,} cells = {int((mndwi[name] > 0).sum()) * CELL_KM2:.2f} km²")

In [ ]:
box = (358000, 3076500, 361000, 3080500)          # the place where the rivers meet, as in D1
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
fig.subplots_adjust(wspace=0.35)
for row, (index, label) in enumerate([(ndwi, "NDWI"), (mndwi, "MNDWI")]):
    for col, (name, date) in enumerate([("before", "29 Oct 2020"), ("after", "11 Nov 2021")]):
        zoom = index[name].rio.clip_box(*box)
        l, b, r, t = zoom.rio.bounds()
        im = axes[row, col].imshow(zoom, extent=[l / 1000, r / 1000, b / 1000, t / 1000], cmap="RdBu", vmin=-1, vmax=1)
        axes[row, col].set_title(f"{label}, {date}")
fig.colorbar(im, ax=axes, label="Index value (blue = above 0)", shrink=0.6)
fig.text(0.01, 0.01, "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.", fontsize=9)
plt.show()

Before the flood only 592 cells (0.24 km²) have an NDWI above 0, and 339 cells (0.14 km²) an MNDWI above 0. After it there are 2,083 and 1,425 cells: 0.83 and 0.57 km². The river spread out in several channels over the new, wide bed of sediment, and the wet ground counts too.

In `clip_box(*box)` the `*` unpacks the four numbers of the tuple into four arguments.

On the zoomed maps, `"RdBu"` is diverging and centred on 0 by `vmin=-1, vmax=1`: blue for water-like cells, red for vegetation, white near 0. In 2020 the river is a thin, broken thread, blue in places. In 2021 a pale band fills the valley floor: the sediment, slightly below 0. Thin blue threads of water run through it. The sediment is pale, not blue: it reflects much NIR and SWIR, so these indices mark it as dry ground.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** What is the highest MNDWI value after the flood? Use `.max()` and `float()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
float(mndwi["after"].max())
```

About 0.57. Clear, deep water often reaches values well above 0.5; here even the best cells mix water with gravel.
</details>

**Q2 · two lines:** Do the two water indices agree? Count the cells of the after image where **both** NDWI and MNDWI are above 0 (combine two masks with `&`, C5), and print the count and its area in km².

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
both = (ndwi["after"] > 0) & (mndwi["after"] > 0)
print(int(both.sum()), round(int(both.sum()) * CELL_KM2, 2))
```

935 cells (0.37 km²): 935 of the 1,425 MNDWI cells, while NDWI marks 2,083 cells in total. The two indices agree on the clearest water and disagree at the edges of channels and on wet gravel. When indices disagree, the cell is probably mixed.
</details>

**Q3 · three lines:** Sample MNDWI after along a line across the valley: at northing 3,078,000 m, every 40 m from easting 359,300 to 360,100 m. Use `.sel(x=..., y=3078000, method="nearest")` with a NumPy array of eastings, and print the values rounded to 2 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
eastings_m = np.arange(359300, 360101, 40)
cross = mndwi["after"].sel(x=eastings_m, y=3078000, method="nearest")
print(np.round(cross.values, 2))
```

From west to east the values climb from about −0.6 on the vegetated bank to between about −0.4 and −0.1 on the sediment bed. One value, about 0.27, is above 0, where a water channel crosses the line. Then they fall again to about −0.5 to −0.7 on the eastern side. One line through a map can explain more than the whole map's mean.
</details>

## 6. NDVI and elevation: the DEM on the image grid

Does vegetation change with height? To compare NDVI with elevation, both must be on the **same grid** (C4). The DEM of Part C is in degrees with 30 m cells; the images are in UTM 45N with 20 m cells.

`dem.rio.reproject_match(reference)` resamples the DEM onto the reference grid: same CRS, transform and shape (C4). Elevation is continuous, so use **bilinear** resampling.

Then each cell gets an **elevation band** of 250 m: `(elevation // 250) * 250` gives 750 for 750 to 999 m, 1,000 for 1,000 to 1,249 m, and so on (A2's `//`). A pandas table with one row per cell, and `groupby` (A9), gives the mean NDVI of each band on both dates.

Bands with very few cells give unreliable means, so the example keeps bands with at least 1,000 cells.

In [ ]:
dem = rxr.open_rasterio(DATA / "dem_kathmandu_melamchi.tif", masked=True).squeeze("band", drop=True)
dem_grid = dem.rio.reproject_match(ndvi_before, resampling=Resampling.bilinear)
print(dem_grid.rio.crs, dem_grid.shape, dem_grid.rio.resolution())
print("elevation in the image box:", round(float(dem_grid.min())), "to", round(float(dem_grid.max())), "m")

cells = pd.DataFrame({
    "elev_m": dem_grid.values.ravel(),
    "elev_band_m": ((dem_grid.values // 250) * 250).ravel(),
    "ndvi_before": ndvi_before.values.ravel(),
    "ndvi_after": ndvi_after.values.ravel(),
}).dropna()

by_band = cells.groupby("elev_band_m").agg(cells=("ndvi_before", "size"),
                                           before=("ndvi_before", "mean"),
                                           after=("ndvi_after", "mean"))
by_band["change"] = by_band["after"] - by_band["before"]
by_band = by_band[by_band["cells"] >= 1000]
by_band.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(by_band.index + 125, by_band["before"], marker="o", color="seagreen", label="29 October 2020")
ax.plot(by_band.index + 125, by_band["after"], marker="o", color="darkorange", label="11 November 2021")
ax.set_xlabel("Elevation (m, middle of each 250 m band)")
ax.set_ylabel("Mean NDVI")
ax.set_title("Mean NDVI by elevation, Melamchi valley")
ax.legend()
plt.show()

The DEM now has the image grid: EPSG:32645, 1,006 × 702 cells of 20 m. Inside the image box the land rises from about 765 m to about 3,503 m.

The table has one row per 250 m band from 750 m to 3,250 m (the 3,500 m band, with only a handful of cells, was dropped). Mean NDVI rises with height, from about 0.65 in the lowest band (750 to 999 m) to about 0.86 between 2,500 and 3,000 m, then dips a little: the valley floor holds the river, fields and villages, the upper slopes mostly forest.

Mean NDVI is lower after than before in every band, by between about 0.015 and 0.034 in all bands but one. In the lowest band it fell by about 0.075, more than twice as much as in any other: that band contains the valley floor where the flood left its sediment. The small drop everywhere else is a reminder from D1: the dates, sun angle and satellites differ, so some change is not on the ground.

`by_band.index + 125` plots each band at its middle. `dropna()` removes cells without NDVI on either date.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How large is the lowest elevation band (750 m) in km²? Use `.loc` on `by_band` and multiply by `CELL_KM2`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
by_band.loc[750, "cells"] * CELL_KM2
```

About 16.8 km² (42,016 cells). It is a small part of the image, which is why the flood hardly moves the whole-image mean.
</details>

**Q2 · two lines:** Which elevation band lost the most NDVI? Use `idxmin()` on the `change` column, and print it with the change of that band rounded to 3 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
worst = by_band["change"].idxmin()
print(worst, round(by_band.loc[worst, "change"], 3))
```

The 750 m band (750 to 999 m), with a change of about −0.075. The flood ran along the lowest ground of the valley.
</details>

**Q3 · three lines:** Look closer at the valley floor: use 100 m bands instead of 250 m, for the cells of `cells` below 1,200 m only, and print the mean NDVI before and after of each band. The `elev_m` column holds each cell's elevation.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
low = cells[cells["elev_m"] < 1200].copy()
low["band_100m"] = (low["elev_m"] // 100) * 100
print(low.groupby("band_100m")[["ndvi_before", "ndvi_after"]].mean().round(3))
```

The 700 m band (765 to 799 m, the lowest ground) falls from about 0.32 to about 0.12, and the 800 m band from about 0.61 to about 0.52. The bands from 900 m up lose only about 0.03 to 0.05. `.copy()` makes `low` a table of its own, so adding a column does not touch `cells`. A finer split shows that the loss sits on the lowest ground, where the river runs.
</details>

## 7. Saving an index as a GeoTIFF

An index is worth saving: other programs, such as QGIS, can open it, and later scripts can read it back instead of computing it again. Two choices matter:

- **dtype `float32`**: indices are decimals from −1 to 1. Stored in `uint16` they would be rounded to 0 or 1, and negatives would be impossible.
- **nodata `NaN`**: a float file can say "no value" with `NaN`, which never collides with a real index value.

The quickest route copies the **profile** of the source image (C1), then updates what changed: one band instead of ten, `float32`, nodata `NaN`. CRS, transform, size and compression come along unchanged. `dst.set_band_description(1, "NDVI")` names the band, as the source file named its bands.

In [ ]:
with rasterio.open(after_path) as src:
    profile = src.profile                          # uint16, 10 bands, nodata 0
print(profile["dtype"], profile["count"], profile["nodata"])

profile.update(count=1, dtype="float32", nodata=np.nan)
ndvi_path = OUT / "d2_ndvi_after.tif"
with rasterio.open(ndvi_path, "w", **profile) as dst:
    dst.write(ndvi_after.values.astype("float32"), 1)
    dst.set_band_description(1, "NDVI (B08 - B04) / (B08 + B04), 11 November 2021")

with rasterio.open(ndvi_path) as src:
    print(src.count, src.dtypes, src.nodata, src.crs, src.res)
    print(src.descriptions)
    back = src.read(1)
print("NaN cells:", np.isnan(back).sum(), "| same values:", np.allclose(back, ndvi_after.values, equal_nan=True))
print("file size:", round(ndvi_path.stat().st_size / 1000000, 2), "MB")

The source profile says `uint16`, 10 bands and nodata 0. After `update`, the new file has 1 band of `float32` with nodata `nan`, in EPSG:32645 with 20 m cells, and its band is named.

Read back, it holds the 128 `NaN` cells of the after image, and `np.allclose(..., equal_nan=True)` confirms the values survived; `equal_nan=True` counts `NaN` = `NaN` as equal. The file is about 2.4 MB: decimals compress less well than whole numbers.

Save the formula and the date with the data, as the description does. A file called "ndvi.tif" with no date is a puzzle six months later.

### Practice

**Q1 · one line:** Print the profile entries that the copy kept from the source: `profile["crs"]` and `profile["transform"]`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(profile["crs"], profile["transform"])
```

EPSG:32645 and a transform with 20 m cells starting at 350,260 m east and 3,095,920 m north: the grid of the images, so the NDVI file lines up with them exactly.
</details>

**Q2 · two lines:** Save NDVI **before** in the same way, as `d2_ndvi_before.tif`, with its own description. Reuse `profile`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(OUT / "d2_ndvi_before.tif", "w", **profile) as dst:
    dst.write(ndvi_before.values.astype("float32"), 1)
    dst.set_band_description(1, "NDVI (B08 - B04) / (B08 + B04), 29 October 2020")
```

The same profile fits, because both images share one grid. With both files saved, any later script, or QGIS, can compare the two dates without recomputing them.
</details>

**Q3 · three lines:** Check the saved before file: open it with rioxarray (`masked=True`), and print its dtype, its band description from `.attrs["long_name"]`, and its mean rounded to 3 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
check = rxr.open_rasterio(OUT / "d2_ndvi_before.tif", masked=True)
print(check.dtype, check.attrs["long_name"])
print(round(float(check.mean()), 3))
```

`float32`, the description you wrote, and a mean of about 0.765, the same as in step 3. Reading a file back is the only proof that it was written correctly.
</details>

## 8. When it breaks: `NaN` in a `uint16` file

You want to save NDVI after quickly. You copy the profile and set nodata to `NaN`, but forget the dtype. Run the cell.

In [ ]:
with rasterio.open(after_path) as src:
    profile_bad = src.profile
profile_bad.update(count=1, nodata=np.nan)          # dtype not changed!
with rasterio.open(OUT / "d2_broken.tif", "w", **profile_bad) as dst:
    dst.write(ndvi_after.values, 1)

The last line of the error says:

`ValueError: Given nodata value, nan, is beyond the valid range of its data type, uint16.`

- **ValueError**: a value that cannot be used.
- **Given nodata value, nan**: the `NaN` you set as nodata.
- **its data type, uint16**: the profile still describes the source image, whose bands are whole numbers from 0 to 65,535. `NaN` is a float, so a `uint16` file cannot hold it.

rasterio stops you here, which is lucky. Had you set nodata to 0 instead, the file would have been written, and every NDVI value would have been cut to a whole number: almost all of them would become 0 (the decimals are dropped).

rasterio may already have created a small, empty `d2_broken.tif` in `OUT`; you can delete it.

The fix: update the dtype together with the nodata value, as in step 7: `profile.update(count=1, dtype="float32", nodata=np.nan)`. Whenever you copy a profile, check every entry against the data you are about to write.

**Try it:** fix the cell.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A bare-ground index: NDBI

The **Normalised Difference Built-up Index** (NDBI, Zha, Gao and Ni, 2003) is (SWIR 1 − NIR) / (SWIR 1 + NIR), for Sentinel-2 (B11 − B08) / (B11 + B08). It was designed for built-up land, but bare rock and sediment, which reflect more SWIR than NIR, are positive too. Compute it for both dates, print the area above 0 in km², and draw the after map with a diverging colour map centred on 0.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
ndbi_before = normalized_difference(before.sel(band="B11"), before.sel(band="B08"))
ndbi_after = normalized_difference(after.sel(band="B11"), after.sel(band="B08"))
for name, ndbi in [("before", ndbi_before), ("after", ndbi_after)]:
    print(name, "NDBI > 0:", round(int((ndbi > 0).sum()) * CELL_KM2, 2), "km²")

fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(ndbi_after, extent=extent_km, cmap="PuOr_r", vmin=-0.8, vmax=0.8)
fig.colorbar(im, ax=ax, label="NDBI (B11 - B08) / (B11 + B08)", shrink=0.7)
ax.set_title("NDBI, 11 November 2021")
fig.text(0.01, 0.01, "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.", fontsize=8)
plt.show()
```

About 12.08 km² is above 0 before and about 26.49 km² after. The flood strip is clearly positive, but so are many dry fields on the slopes in November, which more than doubles the area. An index marks a spectral property, not a land-cover class: it cannot tell flood sediment from a harvested field. `vmin=-0.8, vmax=0.8` keeps the colour map centred on 0 while using its colours where the values are.
</details>

### Review 2 · One table of index statistics

Write a function `summarise(index)` that returns a dictionary with the mean, the median and the area in km² of cells above 0. Use it in a loop over NDVI, NDWI and MNDWI on both dates, and build a pandas table with one row per index and date.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
def summarise(index):
    """Mean, median and area above 0 (km²) of an index map."""
    return {"mean": float(index.mean()), "median": float(index.median()),
            "area_above_0_km2": int((index > 0).sum()) * CELL_KM2}

rows = []
for date in ["before", "after"]:
    maps = {"NDVI": ndvi_before if date == "before" else ndvi_after, "NDWI": ndwi[date], "MNDWI": mndwi[date]}
    for name, index in maps.items():
        rows.append({"index": name, "date": date, **summarise(index)})
print(pd.DataFrame(rows).round(3))
```

Six rows. NDVI's median is above 0.8 on both dates, and nearly all of the image is above 0. The water indices have negative means (about −0.7 for NDWI, −0.6 for MNDWI) and tiny areas above 0 (at most 0.833 km²), several times larger after the flood. `**summarise(index)` unpacks the dictionary into the row, next to the index name and date.
</details>

### Review 3 · Where the rivers meet, before and after

Cut the box `box` (step 5) out of `ndvi_before` and `ndvi_after` with `clip_box`. For each date, print the mean NDVI and the area in km² with NDVI below 0.3. Then draw both clips side by side with `RdYlGn`, `vmin=-1`, `vmax=1` and the dates in the titles.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
fig.subplots_adjust(wspace=0.35)
for ax, ndvi, date in zip(axes, [ndvi_before, ndvi_after], ["29 October 2020", "11 November 2021"]):
    zoom = ndvi.rio.clip_box(*box)
    print(date, "| mean NDVI:", round(float(zoom.mean()), 3), "| NDVI < 0.3:", round(int((zoom < 0.3).sum()) * CELL_KM2, 2), "km²")
    l, b, r, t = zoom.rio.bounds()
    im = ax.imshow(zoom, extent=[l / 1000, r / 1000, b / 1000, t / 1000], cmap="RdYlGn", vmin=-1, vmax=1)
    ax.set_title(f"NDVI, {date}")
fig.colorbar(im, ax=axes, label="NDVI", shrink=0.7)
plt.show()
```

In this 3 km × 4 km box the mean NDVI falls from about 0.675 to about 0.618, and the area below 0.3 grows from about 0.67 km² to about 1.57 km². Close to the river, the flood is a large share of the land, so it moves the statistics far more than over the whole image.
</details>

### Review 4 · Save MNDWI with rioxarray

`mndwi["after"]` is a DataArray with a CRS, so rioxarray can save it in one line. Write it as `d2_mndwi_after.tif` with `.rio.write_nodata(np.nan).rio.to_raster(...)`, then open the file with rasterio and print the dtype, nodata, CRS and the number of cells above 0.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
mndwi_path = OUT / "d2_mndwi_after.tif"
mndwi["after"].rio.write_nodata(np.nan).rio.to_raster(mndwi_path)

with rasterio.open(mndwi_path) as src:
    print(src.dtypes, src.nodata, src.crs)
    saved = src.read(1)
print("cells above 0:", (saved > 0).sum())
```

`float32`, nodata `nan`, EPSG:32645, and 1,425 cells above 0, as in step 5. This route works because `open_s2` cleared the file's old attributes: with `scale_factor` 0.0001 still attached, the new file would tell other programs to scale the index again.
</details>

## Recap

- Band maths on `uint16` bands **wraps around** instead of going negative. Convert to `float32` (and to reflectance) **before** calculating, and turn nodata into `NaN` first, or nodata cells pose as index values of exactly 1 or −1.
- A **normalised difference** (A − B) / (A + B) lies between −1 and 1. One function, with zero sums replaced by `NaN`, serves every index: **NDVI** (B08, B04), **NDWI** (B03, B08), **MNDWI** (B03, B11), **NDBI** (B11, B08).
- Map indices with a **diverging** colour map centred on 0 (`vmin=-1, vmax=1` or another symmetric pair), and give both dates the same limits. Always name the formula with the index.
- A **threshold** turns an index into a mask; cells × 0.0004 km² give an area. The threshold is your choice: check the histogram, say which one you used, and expect the area to move with it.
- Put other rasters on the image grid with `reproject_match` (bilinear for the DEM) to compare by elevation with `groupby`. Save indices as **`float32` GeoTIFFs with nodata `NaN`**, updating the copied profile's dtype too.

### Words you learned

| Word | Meaning |
| --- | --- |
| band maths | Arithmetic with bands, cell by cell |
| unsigned integer (`uint16`) | A whole number with no sign: 0 to 65,535; negative results wrap around |
| spectral index | One number per cell, calculated from several bands, that highlights one kind of surface |
| normalised difference | (A − B) / (A + B): a difference scaled to lie between −1 and 1 |
| NDVI | Normalised Difference Vegetation Index, (NIR − red) / (NIR + red) |
| NDWI | Normalised Difference Water Index (McFeeters), (green − NIR) / (green + NIR) |
| MNDWI | Modified NDWI (Xu), (green − SWIR 1) / (green + SWIR 1) |
| NDBI | Normalised Difference Built-up Index, (SWIR 1 − NIR) / (SWIR 1 + NIR) |
| guard (division) | A check that avoids dividing by zero, here by replacing zero sums with `NaN` |
| diverging colour map | A colour map with two contrasting ends and a neutral middle, for values above and below a centre |
| threshold | A chosen value that splits cells into two groups |
| log scale | An axis on which each power of ten takes the same length |
| elevation band | A range of elevations, here 250 m wide, used to group cells |
| profile (rasterio) | The dictionary that describes a raster file: driver, dtype, count, CRS, transform, nodata |

## Next

**D3 · Change detection: the 2021 Melamchi flood:** subtract the before index from the after index, choose and test thresholds, clean the change mask, and map and measure the flood strip honestly.

<!-- pygile-nav -->
**PyGILE Course** · lesson 31 of 32 · [Course home](../README.md)  
← Previous: [D1 · Satellite images and spectral bands: the Melamchi valley from space](D01_Satellite_images_and_spectral_bands.ipynb) · Next: [D3 · Change detection: the 2021 Melamchi flood](D03_Change_detection_Melamchi_flood.ipynb) →